# How the CodeBERT training data was cleaned and prepared

**Why this notebook exists.** The question *"how did you clean and prepare your data?"* needs a clear, repeatable answer. This notebook performs the whole preparation of the training data **step by step**, shows what each rule removes, draws the numbers, and at the end **proves that the result is exactly the file that was used for training** (byte for byte).

**The data.** Requirement / code pairs written by LLMs (30 hand-designed domains plus generated batches). Each item = one requirement, the code that implements it, and 2 *hard negatives* (code that looks similar but does something else). Raw text from an LLM is not safe to train on directly, so every item goes through rules first.

**The code is the same as `scripts/prepare_generated_pairs.py`**; here it is split into cells with an explanation above each one. The last part of the notebook explains the preparation of the **real, expert-labelled** projects (`scripts/load_dataset.ipynb`) that the cross-validation uses.

## 1. Environment

Finds the project folder and sets the plot style. No GPU or model is needed for this notebook.

In [ ]:
import os, sys, json, re, csv, glob, random, hashlib
from pathlib import Path
from collections import Counter
ROOT = Path.cwd().resolve()
while not (ROOT / 'agents').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent                                  # find the project folder from wherever the notebook runs
os.chdir(ROOT); sys.path.insert(0, str(ROOT)); csv.field_size_limit(sys.maxsize)
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
SYN, OUT_CHECK = ROOT / 'data' / 'generated_pairs', ROOT / 'data' / 'generated_pairs_clean_check'
print('project folder:', ROOT)

## 2. The raw sources

Raw items come from folders of JSON files (`user_domains`: 30 domains of 65 items designed by hand; `deepseek`, `claude`: generated batches) and from CSV batches (`gemini`). The graph shows how many raw items each source gives. Nothing is cleaned yet.

In [ ]:
JSON_FOLDERS = ('user_domains', 'deepseek', 'claude')

def load_json_items(report):
    items = []
    for f in sorted(p for folder in JSON_FOLDERS for p in glob.glob(str(SYN / folder / '*.json'))):
        group = f"{Path(f).parent.name}/{Path(f).stem}"
        for i, x in enumerate(json.loads(Path(f).read_text(encoding='utf-8'))):
            if not isinstance(x, dict) or not x.get('requirement') or not x.get('code'):
                report['bad_json_item'] += 1; continue
            items.append({'group': group, 'id': f'{group}#{i}', 'requirement': x['requirement'], 'code': x['code'], 'hard': list(x.get('hard_negatives') or [])})
    return items

def load_csv_items(report):
    grouped = {}
    for f in sorted(glob.glob(str(SYN / 'gemini' / '*.csv'))):
        with open(f, encoding='utf-8', newline='') as fh:
            for row in csv.DictReader(fh): grouped.setdefault(row.get('requirement_id', ''), []).append(row)
    items = []
    for rid, rows in grouped.items():
        pos = [r for r in rows if r.get('kind') == 'pos' and r.get('label') == '1']; hard = [r for r in rows if r.get('kind') == 'hard' and r.get('label') == '0']
        if len(pos) != 1 or len(hard) != 2: report['csv_incomplete_item'] += 1; continue
        items.append({'group': 'gemini', 'id': rid, 'requirement': pos[0]['requirement_text'], 'code': pos[0]['code_text'], 'hard': [h['code_text'] for h in hard]})
    return items

report = Counter()
raw_items = load_json_items(report) + load_csv_items(report)
by_source = Counter(it['group'].split('/')[0] for it in raw_items)
print('raw items per source:', dict(by_source), '| total', len(raw_items))
fig, ax = plt.subplots(figsize=(6, 2.8)); ax.bar(list(by_source), list(by_source.values()), color='#4c78a8')
for i, v in enumerate(by_source.values()): ax.text(i, v + 15, str(v), ha='center')
ax.set_ylabel('raw items'); ax.set_title('Raw items by source'); plt.tight_layout(); plt.show()

### One raw item

A requirement, the code that implements it (with its `package`, `import` lines and header comments), and the first hard negative.

In [ ]:
it = raw_items[0]
print('REQUIREMENT:', it['requirement'], '\n\nRIGHT CODE (raw):\n', it['code'][:500], '\n...\n\nHARD NEGATIVE 1 (raw):\n', it['hard'][0][:300], '\n...')

## 3. The cleaning rules

Each rule exists for a reason. **A bad item is dropped as a whole**, never half-kept.

1. **Boilerplate removal.** `package`, `import` lines and licence headers carry no meaning about *what* the code does, and every file in a domain shares them. They are removed with the project's own `strip_boilerplate`, and spaces are collapsed.
2. **Code must be real.** After cleaning, the code must still contain text and its brackets must balance. A class cut off in the middle (typical LLM problem) is dropped.
3. **No label-revealing words.** If a class were called `WrongLoginHandler` or a requirement said "decoy", the model could learn the word instead of the meaning. Class names with *correct, wrong, fake, alternative, negative, decoy, variant, sample, test, dummy* are rejected (in requirements only the clearly shortcut words are checked, because "negative balance" is legitimate).
4. **A negative must be different.** A hard negative that is identical to the right code is not a negative.
5. **No duplicates.** The same requirement (case-insensitive) or the same right code is kept once, so the model is not tested on what it trained on.

In [ ]:
from agents.traceability_agent.preprocess import strip_boilerplate
from shared.schemas.traceability import Artifact

FORBIDDEN = re.compile(r"\b(correct|wrong|fake|alternative|negative|decoy|variant|sample|test|dummy)\b", re.IGNORECASE)
FORBIDDEN_REQ = re.compile(r"\b(correct|wrong|fake|alternative|decoy|variant|dummy)\b", re.IGNORECASE)
CLASS_NAME = re.compile(r"\bclass\s+(\w+)")

def label_revealing_name(code):
    """True if the declared class name itself says right/wrong (the shortcut we removed)."""
    return any(FORBIDDEN.search(name) for name in CLASS_NAME.findall(code))

def balanced(code):
    pairs, stack, in_str, prev = {')': '(', '}': '{', ']': '['}, [], False, ''
    for ch in code:
        if ch == '"' and prev != '\\': in_str = not in_str
        elif not in_str:
            if ch in '({[': stack.append(ch)
            elif ch in pairs and (not stack or stack.pop() != pairs[ch]): return False
        prev = ch
    return not stack and not in_str

def clean_code(raw):
    return ' '.join(strip_boilerplate(Artifact(id='x', type='code', text=raw)).split())

print('BEFORE (raw, first 260 characters):\n', raw_items[0]['code'][:260].replace('\n', ' '))
print('\nAFTER clean_code:\n', clean_code(raw_items[0]['code'])[:260])
print('\nbalanced braces?', balanced(clean_code(raw_items[0]['code'])), '| label-revealing class name?', label_revealing_name(clean_code(raw_items[0]['code'])))

## 4. Apply the rules to every item

This is the real cleaning pass. For each item the first rule that fails drops it, and the reason is counted (and one example per reason is kept so it can be shown below).

In [ ]:
report = Counter(); load_json_items(report); load_csv_items(report)          # reload to count unreadable items again
report['items_read'] = len(raw_items)
kept, seen_req, seen_code, dropped_examples = [], set(), set(), {}

def drop(reason, item):
    report[reason] += 1; dropped_examples.setdefault(reason, item)

for it in raw_items:
    req = ' '.join(it['requirement'].split())
    if FORBIDDEN_REQ.search(req): drop('forbidden_requirement', it); continue
    pos_code = clean_code(it['code']); negs = [clean_code(h) for h in it['hard']]
    if not pos_code or not balanced(pos_code): drop('bad_positive_code', it); continue
    ok = True
    for n in negs:
        if not n or not balanced(n) or label_revealing_name(n) or n == pos_code: ok = False; break
    if label_revealing_name(pos_code) or not ok: drop('bad_negative_or_forbidden_code', it); continue
    if req.lower() in seen_req: drop('duplicate_requirement', it); continue
    if pos_code in seen_code: drop('duplicate_positive_code', it); continue
    seen_req.add(req.lower()); seen_code.add(pos_code)
    kept.append({'group': it['group'], 'id': it['id'], 'req': req, 'pos': pos_code, 'negs': negs})
report['items_kept'] = len(kept)
reasons = {k: v for k, v in report.items() if k not in ('items_read', 'items_kept')}
print(f"read {report['items_read']} -> kept {report['items_kept']} (dropped {report['items_read'] - report['items_kept']})"); print('reasons:', reasons)

labels = ['read', *reasons, 'kept']; values = [report['items_read'], *[-v for v in reasons.values()], report['items_kept']]
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.bar(range(len(labels)), [abs(v) for v in values], color=['#4c78a8'] + ['#d1495b'] * len(reasons) + ['#2f6f4e'])
for i, v in enumerate(values): ax.text(i, abs(v) + 20, str(abs(v)), ha='center')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=15, ha='right'); ax.set_ylabel('items'); ax.set_title('What the cleaning removes'); plt.tight_layout(); plt.show()

### What was dropped

One example for each reason that removed something, so you can see that the rules catch real problems.

In [ ]:
for reason, it in dropped_examples.items():
    print(f"--- dropped because: {reason}\nREQUIREMENT: {it['requirement'][:160]}\nCODE (raw): {it['code'][:200].replace(chr(10), ' ')} ...\n")
if not dropped_examples: print('nothing was dropped')

## 5. Split by domain and write the files

The cleaned items are split **by domain** (group), not by item: about 10% of the domains go to the *validation* file (they choose the best training epoch). Splitting by domain means validation measures a new domain, as the real test later does. The seed is fixed, so the split is always the same. Each item becomes 3 rows (1 right code, 2 hard negatives) in the format the training notebook reads. The files are written to a **check folder**, not over the real ones.

In [ ]:
SEED, VALID_FRACTION = 33, 0.10
groups = sorted({k['group'] for k in kept}); rng = random.Random(SEED)
valid_groups = set(rng.sample(groups, max(1, round(len(groups) * VALID_FRACTION)))) if groups else set()
OUT_CHECK.mkdir(parents=True, exist_ok=True)
counts = {'train': Counter(), 'valid': Counter()}
files = {name: (OUT_CHECK / f'{name}.jsonl').open('w', encoding='utf-8') for name in ('train', 'valid')}
try:
    for k in kept:
        split = 'valid' if k['group'] in valid_groups else 'train'
        rows = [(f"{k['id']}-pos", k['pos'], 1, 'pos')] + [(f"{k['id']}-hard{j}", n, 0, 'hard') for j, n in enumerate(k['negs'])]
        for target_id, doc, label, kind in rows:
            files[split].write(json.dumps({'dataset': 'synthetic', 'source_id': k['id'], 'target_id': target_id, 'query': k['req'], 'doc': doc, 'label': label, 'kind': kind}, ensure_ascii=False) + '\n'); counts[split][kind] += 1
finally:
    for fh in files.values(): fh.close()
print('domains:', len(groups), '| validation domains:', len(valid_groups)); print({s: dict(c) for s, c in counts.items()})
fig, ax = plt.subplots(figsize=(5, 2.8)); ax.bar(['train', 'validation'], [counts['train']['pos'], counts['valid']['pos']], color=['#4c78a8', '#e0a030'])
for i, v in enumerate([counts['train']['pos'], counts['valid']['pos']]): ax.text(i, v + 15, str(v), ha='center')
ax.set_ylabel('items'); ax.set_title('Items per file (the 6 test domains are cut from train later)'); plt.tight_layout(); plt.show()

## 6. Proof: this is exactly the file that trained the model

The file that the training notebook reads is `data/generated_pairs_clean/`. If the cleaning is repeatable, the files made just now must be **identical** to it. Their fingerprints (SHA-256) are compared.

In [ ]:
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
all_same = True
for name in ('train.jsonl', 'valid.jsonl'):
    a, b = sha(OUT_CHECK / name), sha(ROOT / 'data' / 'generated_pairs_clean' / name); same = a == b; all_same &= same
    print(f'{name:12s} new {a[:16]}...   used for training {b[:16]}...   {"IDENTICAL" if same else "DIFFERENT"}')
print('\nRESULT:', 'the cleaning is reproducible, byte for byte.' if all_same else 'the files differ: the raw data or the rules changed since the training file was made.')

## 7. The real, expert-labelled projects (used for the cross-validation)

The seven real projects (iTrust, eTour, SMOS, eANCI, LibEST, CM-1, EBT) are public datasets with links labelled by experts. They are prepared in **`scripts/load_dataset.ipynb`**, which reads the raw downloads in `data/raw` and writes `data/processed/<project>/{artifacts, traces, gold}.json`:

* requirements and code are turned into *artifacts* (id, type, text); the expert links become *traces*; every (requirement, file) pair gets a yes/no label in `gold.json`;
* SMOS and eANCI are Italian: requirements and code comments were translated to English by an LLM (`data/translated_en`, with the originals kept and a glossary of identifier words), so the English-language models can read them;
* inside the pipeline, code is cleaned again at retrieval time (`preprocess.py`: boilerplate removed, identifiers split into words).

The graph below is computed from the prepared files.

In [ ]:
PROJECTS = ['itrust', 'etour', 'smos', 'eanci', 'libest', 'cm1', 'ebt']; stats = {}
for p in PROJECTS:
    arts = json.loads((ROOT / 'data' / 'processed' / p / 'artifacts.json').read_text(encoding='utf-8')); traces = json.loads((ROOT / 'data' / 'processed' / p / 'traces.json').read_text(encoding='utf-8'))
    stats[p] = (sum(a['type'] == 'requirement' for a in arts), sum(a['type'] != 'requirement' for a in arts), len(traces)); print(f'{p:7s} {stats[p][0]:4d} requirements  {stats[p][1]:4d} targets  {stats[p][2]:5d} expert links')
x = np.arange(len(PROJECTS)); fig, ax = plt.subplots(figsize=(9, 3.2))
for j, (label, c) in enumerate((('requirements', '#4c78a8'), ('code / design files', '#2f6f4e'), ('expert links', '#d1495b'))): ax.bar(x + (j - 1) * 0.27, [stats[p][j] for p in PROJECTS], 0.27, color=c, label=label)
ax.set_xticks(x); ax.set_xticklabels(PROJECTS); ax.set_yscale('log'); ax.set_ylabel('count (log scale)'); ax.set_title('The prepared real projects'); ax.legend(); plt.tight_layout(); plt.show()

## What to say in the viva

* **Raw data:** requirement / code / 2 hard negatives per item, written by LLMs in 59 domains.
* **Cleaning:** five rules (boilerplate removed, code must be complete, no label-revealing words, negatives must differ, no duplicates); a bad item is dropped as a whole. The notebook shows how many items each rule removes and an example of each.
* **Split:** by domain, with a fixed seed, so validation and test measure new domains.
* **Reproducible:** re-running the cleaning gives exactly the training file (same SHA-256).
* **Real data:** the seven expert-labelled projects are prepared separately (`load_dataset.ipynb`) and used for the cross-validation, where a project is never in the training data of the model tested on it.